# SI — does the fast vectorial model agree with the full physics?

`pyS3M.simulation.vectorial_sampler` simulates the vectorial PSF of a freely rotating dipole fast:
one radial profile at 600 nm, rescaled to each photon's wavelength, sampled photon by photon
(`FigureSI_VectorialScaling.ipynb` shows why that is allowed). Before it becomes the default
simulation model, this notebook checks it against the **reference model**, which uses no lookup
table, rescaling or symmetry:

- the full 2D `DipolePSF` (Fazel et al. 2024, Rev. Mod. Phys. 96, 025003) computed directly at
  **every 1 nm wavelength** of the dye's spectrum, on a finer pupil (`REF_N_PUPIL`) and integrated over
  each pixel with finer sub-sampling (`REF_OVERSAMPLE`), at the exact emitter position;
- each pixel's expected signal is then
  $E_j = N \sum_\lambda p(\lambda)\,\Delta\lambda\;\mathrm{QE}_{t(j)}(\lambda)\;\mathrm{PSF}_\lambda(j)$, with $t(j)$
  pixel $j$'s type.

**Why Poisson per pixel is the exact reference for the noise.** In both models the number of
photons is Poisson($N$), and each photon independently picks a wavelength, a landing pixel and
whether it is detected. Thinning a Poisson process gives **independent Poisson counts per pixel** with
means $E_j$, so reference frames are drawn as Poisson($E_j$) per pixel: the same distribution as
photon-by-photon simulation, without needing a PSF per photon. §2 checks that the fast model's frames
have exactly this statistics (variance = mean, no correlation between pixels).

**Checks**

1. Expected images: the fast model's radial table, and the mean of its photon-by-photon frames,
   against the reference, pixel by pixel.
2. Noise: per-pixel variance / mean and pixel–pixel correlations of the fast model's frames.
3. What is fitted: frames from both models (same background, same camera) are fitted with the
   production fitter (`STANDARD_DATA`), and the distributions of the fitted PSF width, localisation
   error and spectral fingerprint compared (means with standard errors, KS tests), across dyes,
   photon numbers and positions in the Bayer cell.

**Photon number.** $N$ is the number of photons landing in the 14 × 14 px ROI, as in the production
model. The fast model redraws a photon that would land outside (wavelength and position), so the
landed photons have the physical distribution: the PSF's shape is unchanged, and their spectrum is
the emission spectrum weighted by each wavelength's fraction landing in the ROI. The reference does
the same exactly: $E_j = N \sum_\lambda p(\lambda)\,\mathrm{QE}_{t(j)}(\lambda)\,\mathrm{PSF}_\lambda(j) \,/\, \sum_\lambda p(\lambda) \sum_{j'} \mathrm{PSF}_\lambda(j')$. With the dipole PSF only ~70% of the collected light lands in the ROI (printed in §1);
a 20 px ROI would add ~7% of the light but < 1% in position precision, so the ROI stays 14 × 14.

---
## 0 — Setup

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

from pyS3M import IOFunctions, SpectralFunctions, MaskFunctions
from pyS3M.simulation.multicolour import MultiC_Sim_Funcs
from pyS3M.simulation.defocus_psf import DipolePSF
from pyS3M.simulation.vectorial_sampler import (
    RadialDipolePSF, photon_by_photon_photoelectrons, background_photoelectrons, camera_adu)
from pyS3M.ImageAnalysisFunctions import Image_Analysis_Functions, FittingStrategy as IAF_FittingStrategy
from pyS3M.sCMOSFunctions import sCMOS_Functions

IO = IOFunctions.IO_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()
MSF = MultiC_Sim_Funcs()
scmos = sCMOS_Functions()

FAST_MODE = True   # False: more repeats, photon levels and positions (slower; for the SI)
REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_DIR = Path("/media/u2473660/JSBLeeLabBackup/Data/20261029_VectorialValidation") / ("fast" if FAST_MODE else "full")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NA, N_MEDIUM, N_IMMERSION = 1.49, 1.33, 1.515
PIXEL_NM = 69.0
H = W = 14                       # ROI, 7 Bayer unit cells
X_CENTRE = Y_CENTRE = 7.0        # emitter at pixel (7, 7) + offset; pixel centres at integers
BACKGROUND_PHOTONS = 5.0         # per pixel, before detection (production's definition)

DYES = ["ATTO 488", "ATTO 565", "ATTO 647N"]
PHOTON_LEVELS = [500, 5000, 50000] if FAST_MODE else [500, 1000, 2000, 5000, 10000, 20000, 50000]
OFFSETS = [(0.0, 0.0), (0.5, 0.5)] if FAST_MODE else [(0.0, 0.0), (0.25, 0.25), (0.5, 0.5), (1.0, 0.0), (0.5, 1.0)]
N_REPEATS = 500 if FAST_MODE else 5000

# Reference: full 2D DipolePSF at every 1 nm of the spectrum, finer than the fast model's source
REF_N_PUPIL = 1024
REF_OVERSAMPLE = 11
REF_MIN_WEIGHT = 1e-5            # skip wavelengths carrying less than this fraction of the photons
RNG_SEED = 20261029

R_qe, G_qe, B_qe, wavelength = S_F.getpixelefficiency()
pixel_order = ["B", "G", "R"]
qe_curves = np.vstack([B_qe, G_qe, R_qe])
SPECTRA = {d: S_F.get_dye_or_filter_data(names=[d], wavelength=wavelength, dye_or_filter=True)[0] for d in DYES}

masks = M_F.get_masks(size_x=H, size_y=W)
pixel_type = np.full((H, W), -1)
for i, c in enumerate(pixel_order):
    pixel_type[masks[c]] = i
masks_3d = np.dstack([masks[c] for c in pixel_order])
N_CH = len(pixel_order)
bg_qe = MSF.background_pixel_efficiency(wavelength, qe_curves, [1.0, 1.0, 1.0])[pixel_type]

gain = float(np.median(IO.read_tiff(CALIBRATION_DIR / "gain.tif")))
offset = float(np.median(IO.read_tiff(CALIBRATION_DIR / "offset.tif")))
variance = float(np.median(IO.read_tiff(CALIBRATION_DIR / "variance.tif")))
rqe = np.full((H, W), float(np.median(IO.read_tiff(CALIBRATION_DIR / "rqe.tif"))))
readnoise = np.sqrt(variance) / gain
cam = dict(gain=np.full((H, W), gain), offset=np.full((H, W), offset), variance=np.full((H, W), variance))

fast_psf = RadialDipolePSF(NA=NA, n_medium=N_MEDIUM, n_immersion=N_IMMERSION)
ref_dipole = DipolePSF(NA=NA, n_medium=N_MEDIUM, n_immersion=N_IMMERSION, N_pupil=REF_N_PUPIL)
print(f"{len(DYES)} dyes x {len(PHOTON_LEVELS)} photon levels x {len(OFFSETS)} offsets x {N_REPEATS} repeats")

In [ ]:
def spectral_weights(dye):
    '''p(lambda) d(lambda) on the 1 nm grid (trapezoid), normalised; wavelengths below REF_MIN_WEIGHT dropped.'''
    pdf = np.maximum(SPECTRA[dye], 0.0)
    w = pdf * np.gradient(wavelength)
    w /= w.sum()
    keep = w > REF_MIN_WEIGHT
    return wavelength[keep], w[keep] / w[keep].sum()


def roi_patch(patch_fn, lam_um, fx, fy, oversample):
    '''Pixel-integrated PSF over the ROI (H x W), emitter at (X_CENTRE + fx, Y_CENTRE + fy) px. The
    15-px patch is centred on pixel (7, 7), so its first 14 rows and columns are the ROI.'''
    p = patch_fn(lam_um, PIXEL_NM / 1000, 15, oversample, fx * PIXEL_NM / 1000, fy * PIXEL_NM / 1000)
    return np.asarray(p).reshape(15, 15)[:H, :W]


def expected_signal(dye, fx, fy, model):
    '''Expected detected signal photoelectrons per pixel for ONE photon landing in the ROI (multiply by N).'''
    lams, w = spectral_weights(dye)
    qe_at = np.vstack([np.interp(lams, wavelength, qe_curves[t]) for t in range(N_CH)])  # (type, lambda)
    out = np.zeros((H, W))
    in_roi = 0.0   # fraction of all collected photons landing in the ROI
    for k, (lam, wk) in enumerate(zip(lams, w)):
        if model == "reference":
            psf = roi_patch(lambda *a: ref_dipole.pixel_patches(*a)[0], lam / 1000, fx, fy, REF_OVERSAMPLE)
        else:
            psf = roi_patch(fast_psf.pixel_patch, lam / 1000, fx, fy, REF_OVERSAMPLE)
        out += wk * qe_at[:, k][pixel_type] * psf
        in_roi += wk * psf.sum()
    return out / in_roi * rqe   # N counts photons landing in the ROI


def roi_fraction(dye, fx, fy):
    '''Fraction of the collected photons landing in the ROI (reference model, spectrum-weighted).'''
    lams, w = spectral_weights(dye)
    return sum(wk * roi_patch(lambda *a: ref_dipole.pixel_patches(*a)[0], lam / 1000, fx, fy, 5).sum()
               for lam, wk in zip(lams[::5], w[::5])) / w[::5].sum()


t0 = time.time()
E_ref = {(d, o): expected_signal(d, *o, "reference") for d in DYES for o in OFFSETS}
t_ref = time.time() - t0
t0 = time.time()
E_fast = {(d, o): expected_signal(d, *o, "fast") for d in DYES for o in OFFSETS}
print(f"expected images: reference {t_ref:.0f} s, fast radial table {time.time() - t0:.1f} s "
      f"({len(DYES) * len(OFFSETS)} dye x offset)")

---
## 1 — Expected images

For every dye and position: the largest difference between the fast model's expected image (its
radial table, rescaled per wavelength) and the reference, relative to the brightest pixel, and the
same as a number of photoelectrons at the largest photon level. The **MC** columns compare the mean
of `N_MC` photon-by-photon frames of the fast model with the reference, as z-scores: |z| should look
like a standard normal (largest |z| ~3–4 over the 196 pixels, mean z² ≈ 1).

In [ ]:
N_MC_PHOTONS, N_MC = 5000, 4000
rng = np.random.default_rng(RNG_SEED)
rows = []
for (dye, (fx, fy)), ref in E_ref.items():
    fast = E_fast[dye, (fx, fy)]
    mc = photon_by_photon_photoelectrons(fast_psf, SPECTRA[dye], wavelength, qe_curves, pixel_type, N_MC_PHOTONS,
                                         np.full(N_MC, X_CENTRE + fx), np.full(N_MC, Y_CENTRE + fy), PIXEL_NM, rng,
                                         rqe=rqe)
    expected = N_MC_PHOTONS * ref
    z = (mc.mean(axis=0) - expected) / np.sqrt(expected / N_MC)
    rows.append({"dye": dye, "offset": (fx, fy),
                 "max |fast - ref| / peak": np.abs(fast - ref).max() / ref.max(),
                 f"max |fast - ref| at {max(PHOTON_LEVELS):,} photons / pe": max(PHOTON_LEVELS) * np.abs(fast - ref).max(),
                 "detected pe per photon in ROI": ref.sum(),
                 "MC max |z|": np.abs(z).max(), "MC mean z^2": np.mean(z ** 2)})
expected_table = pd.DataFrame(rows)
expected_table.to_csv(OUT_DIR / "expected_images.csv", index=False)
print("Fraction of collected photons landing in the 14 x 14 px ROI (reference model):")
for dye in DYES:
    print(f"  {dye}: " + ", ".join(f"offset {o}: {roi_fraction(dye, *o):.3f}" for o in OFFSETS))
expected_table.round(5)

In [ ]:
dye, off = DYES[1], OFFSETS[-1]
ref, fast = E_ref[dye, off], E_fast[dye, off]
fig, axs = plt.subplots(1, 3, figsize=(9, 2.8))
for ax, img, title, cmap in zip(axs, [ref, fast, (fast - ref) / ref.max()],
                                ["reference (per photon in ROI)", "fast", "(fast - ref) / peak"],
                                ["magma", "magma", "RdBu_r"]):
    im = ax.imshow(img, cmap=cmap); ax.set_title(title, fontsize=9); fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle(f"{dye}, offset {off} px", fontsize=9)
fig.tight_layout()
plt.show()

---
## 2 — Noise: the fast model's frames are Poisson per pixel

Photon-by-photon with a Poisson photon number should give independent Poisson counts per pixel
(variance = mean, no correlation between pixels), the same statistics as the reference frames in §3.
Signal only (no background or camera), one dye and position, `N_MC` frames.

In [ ]:
dye, off = DYES[1], OFFSETS[-1]
mc = photon_by_photon_photoelectrons(fast_psf, SPECTRA[dye], wavelength, qe_curves, pixel_type, N_MC_PHOTONS,
                                     np.full(N_MC, X_CENTRE + off[0]), np.full(N_MC, Y_CENTRE + off[1]),
                                     PIXEL_NM, rng, rqe=rqe).reshape(N_MC, -1).astype(float)
mean, var = mc.mean(axis=0), mc.var(axis=0, ddof=1)
bright = mean > 10
corr = np.corrcoef(mc[:, bright].T)
off_diag = corr[~np.eye(len(corr), dtype=bool)]
print(f"{bright.sum()} pixels with > 10 pe: variance / mean = {np.median(var[bright] / mean[bright]):.3f} (median), "
      f"range {np.min(var[bright] / mean[bright]):.3f}-{np.max(var[bright] / mean[bright]):.3f}; "
      f"expected 1 +- {np.sqrt(2 / (N_MC - 1)):.3f}")
print(f"pixel-pixel correlations: mean {off_diag.mean():+.4f}, sd {off_diag.std():.4f} "
      f"(independent: 0 +- {1 / np.sqrt(N_MC):.4f})")
fig, axs = plt.subplots(1, 2, figsize=(7, 2.8))
axs[0].loglog(mean[bright], var[bright], ".", ms=3); lim = [mean[bright].min(), mean[bright].max()]
axs[0].plot(lim, lim, "k--", lw=1); axs[0].set_xlabel("mean / pe"); axs[0].set_ylabel("variance / pe$^2$")
axs[1].hist(off_diag, bins=60); axs[1].set_xlabel("pixel-pixel correlation")
fig.tight_layout()
plt.show()

---
## 3 — What the fitter sees

For every dye, photon number $N$ and position: `N_REPEATS` frames from each model, each with the
same background (Poisson photons, each pixel's background QE) and the same camera (Ximea calibration
medians), converted back to photoelectrons and fitted with production's `STANDARD_DATA` fit, as for
real data. Compared: fitted PSF width σ, localisation error, and the distance of the fitted
fingerprint from the truth. `diff / SE` is the difference of the means in units of its standard error
(|diff / SE| < 2 for most conditions if the models agree); `KS p` tests the whole distributions.

In [ ]:
IAF = Image_Analysis_Functions(readnoise=readnoise)


def to_pe(adu):
    return IO.convert_to_photoelectrons(np.asarray(adu, float), gain_map=cam["gain"], offset_map=cam["offset"], rqe=rqe)


def fit_stack(adu):
    '''Production STANDARD_DATA fit of each frame (as FigureSI_Factorisation); NaN rows where rejected.'''
    data = to_pe(adu)
    smoothed = scmos.gaussian_filter_stack(data, sigma=1.5)
    weights = 1.0 / (np.clip(smoothed, 0, None) + 1.0 + readnoise ** 2)
    n = len(data)
    fit, _ = IAF.fit_puncta_method(list(data), list(smoothed), list(weights), [(0.0, 0.0)] * n, list(range(n)),
                                   strategy=IAF_FittingStrategy.STANDARD_DATA, masks=[masks_3d] * n)
    return fit


def true_fingerprint(dye):
    lams, w = spectral_weights(dye)
    q = np.array([np.sum(w * np.interp(lams, wavelength, qe_curves[t])) for t in range(N_CH)])
    return q / q.sum()


def metrics(fit, x0, y0, truth):
    amps = fit[:, 4 + N_CH:4 + 2 * N_CH]
    fp = amps / amps.sum(axis=1, keepdims=True)
    return {"sigma / nm": 0.5 * (fit[:, 2] + fit[:, 3]) * PIXEL_NM,
            "position error / nm": np.hypot(fit[:, 0] - x0, fit[:, 1] - y0) * PIXEL_NM,
            "fingerprint distance": np.linalg.norm(fp - truth, axis=1)}


rows = []
t0 = time.time()
for i_d, dye in enumerate(DYES):
    truth = true_fingerprint(dye)
    for n_photons in PHOTON_LEVELS:
        for (fx, fy) in OFFSETS:
            rng = np.random.default_rng([RNG_SEED, i_d, n_photons, int(4 * fx), int(4 * fy)])
            x0, y0 = X_CENTRE + fx, Y_CENTRE + fy
            sig = {"reference": rng.poisson(n_photons * E_ref[dye, (fx, fy)], size=(N_REPEATS, H, W)),
                   "fast": photon_by_photon_photoelectrons(fast_psf, SPECTRA[dye], wavelength, qe_curves, pixel_type,
                                                           n_photons, np.full(N_REPEATS, x0), np.full(N_REPEATS, y0),
                                                           PIXEL_NM, rng, rqe=rqe)}
            m = {}
            for model, s in sig.items():
                pe = s + background_photoelectrons(BACKGROUND_PHOTONS, bg_qe, N_REPEATS, rng, rqe=rqe)
                m[model] = metrics(fit_stack(camera_adu(pe, rng=rng, **cam)), x0, y0, truth)
            for metric in m["reference"]:
                a, b = (v[metric][np.isfinite(v[metric])] for v in (m["reference"], m["fast"]))
                se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
                rows.append({"dye": dye, "N": n_photons, "offset": (fx, fy), "metric": metric,
                             "reference": a.mean(), "fast": b.mean(), "diff / SE": (b.mean() - a.mean()) / se,
                             "rel. diff / %": 100 * (b.mean() / a.mean() - 1), "KS p": ks_2samp(a, b).pvalue,
                             "fits ok (ref, fast)": (len(a), len(b))})
    print(f"{dye} done ({time.time() - t0:.0f} s)")
fits = pd.DataFrame(rows)
fits.to_csv(OUT_DIR / "fitted_comparison.csv", index=False)

In [ ]:
print("Largest |diff / SE| per metric, and how many conditions exceed 2 and 3 (expected by chance: "
      f"~5% and ~0.3% of {len(fits) // 3}):")
for metric, g in fits.groupby("metric"):
    print(f"  {metric:22s} max |diff/SE| {g['diff / SE'].abs().max():.2f};  >2: {(g['diff / SE'].abs() > 2).sum()};  "
          f">3: {(g['diff / SE'].abs() > 3).sum()};  max |rel. diff| {g['rel. diff / %'].abs().max():.2f} %;  "
          f"min KS p {g['KS p'].min():.3f}")

fig, axs = plt.subplots(1, 3, figsize=(11, 3))
for ax, (metric, g) in zip(axs, fits.groupby("metric")):
    for dye, gd in g.groupby("dye"):
        ax.semilogx(gd["N"], gd["diff / SE"], "o", ms=4, label=dye)
    ax.axhspan(-2, 2, color="0.9"); ax.axhline(0, color="k", lw=0.5)
    ax.set_title(metric, fontsize=9); ax.set_xlabel("N photons in ROI")
axs[0].set_ylabel("(fast - reference) / SE"); axs[0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUT_DIR / "fitted_comparison.svg", bbox_inches="tight")
plt.show()
fits.round(4)

---
## Verdict

The fast model can replace the reference if (§1) its expected images differ from the reference by
much less than the shot noise at the highest photon level, (§2) its frames are Poisson per pixel,
and (§3) the fitted σ, localisation error and fingerprint distance agree within statistical error
(|diff / SE| > 2 in about 1 in 20 conditions, > 3 in almost none). If so, it becomes the default
simulation model.